In [1]:
import os

In [2]:
%pwd

'e:\\data science\\Deep_learning_project\\kidney_disease_classification\\research'

In [3]:
import os

os.chdir('E:/data science/Deep_learning_project/kidney_disease_classification')

In [4]:
%pwd

'E:\\data science\\Deep_learning_project\\kidney_disease_classification'

In [5]:
import tensorflow as tf

In [6]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

In [7]:
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/Vasunavadiya90/kidney_disease_classification_project.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="vasunavadiya90"
os.environ["MLFLOW_TRACKING_PASSWORD"]="8e4a86cd529c5e5afdf2f7ac2874b2c078102fdb"

In [8]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [9]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [10]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/kidney-ct-scan-image",
            mlflow_uri="https://dagshub.com/Vasunavadiya90/kidney_disease_classification_project.mlflow",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config




In [8]:
!pip install mlflow

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import tensorflow as tf
from pathlib import Path
from urllib.parse import urlparse
import mlflow

In [12]:
import tensorflow as tf
from pathlib import Path
from urllib.parse import urlparse
import mlflow

class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    
    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )


    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = model.evaluate(self.valid_generator)
        self.save_score()

    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    
    def log_into_mlflow(self):
        # Set the tracking URI for logging experiments, params, and metrics
        mlflow.set_tracking_uri(self.config.mlflow_uri)
        mlflow.set_registry_uri(self.config.mlflow_uri)
        
        # Set or create experiment
        mlflow.set_experiment("kidney_disease_classification")
        
        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics(
                {"loss": self.score[0], "accuracy": self.score[1]}
            )
            # Log model artifacts without attempting remote registration
            mlflow.keras.log_model(self.model, "model")

In [14]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
   raise e

Found 139 images belonging to 2 classes.
9/9 ━━━━━━━━━━━━━━━━━━━━ 39s 5s/step - accuracy: 1.0000 - loss: 0.0265


2026/06/17 20:55:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/17 20:55:33 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


🏃 View run peaceful-mouse-379 at: https://dagshub.com/Vasunavadiya90/kidney_disease_classification_project.mlflow/#/experiments/0/runs/4b3d991fcbb145678dc4c075c15b8026
🧪 View experiment at: https://dagshub.com/Vasunavadiya90/kidney_disease_classification_project.mlflow/#/experiments/0
